In [1]:
import os
import io
import argparse
import shutil
import glob
import pandas as pd
from io import BytesIO
import fitz # PyMuPDF package 
import json

from google.cloud import aiplatform
from google.cloud import storage
from pypdf import PdfReader

# Vertex AI
from google import genai
from google.genai import types
from google.genai.types import Content, Part, GenerationConfig, ToolConfig
from google.genai import errors
import time
from semantic_splitter import SemanticChunker

In [2]:
import hashlib

In [3]:
import chromadb

In [4]:
CHROMADB_HOST = "llm-rag-chromadb"
CHROMADB_PORT = 8000

In [5]:
gcp_project = "apcomp215-group88"
bucket_name = "88-data"
EMBEDDING_MODEL = "text-embedding-004"
source_blob_name = "RAG_taxtbook/hpi-sec-sser.pdf" 
OUTPUT_FOLDER = "RAG_outputs"

GCP_LOCATION = "us-central1"
EMBEDDING_DIMENSION = 256
GENERATIVE_MODEL = "gemini-2.0-flash-001"
INPUT_FOLDER = "input-datasets"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:/Users/718/Desktop/llm-service-account.json"
llm_client = genai.Client(vertexai=True, project=gcp_project, location="us-central1")


In [11]:
CHROMADB_HOST = "llm-rag-chromadb"
CHROMADB_PORT = 8000
gcp_project = "apcomp215-group88"
bucket_name = "88-data"
EMBEDDING_MODEL = "text-embedding-004"
INPUTE_FOLDER = 'books'
OUTPUT_FOLDER = "RAG_outputs"
GCP_LOCATION = "us-central1"
EMBEDDING_DIMENSION = 256
GENERATIVE_MODEL = "gemini-2.0-flash-001"
chunk_file = "semantic-chunks-policy.jsonl"
prefix = "Mock_policy_RAG/"
storage_client = storage.Client()
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(INPUTE_FOLDER, exist_ok=True)

bucket = storage_client.bucket(bucket_name)
blobs = bucket.list_blobs(prefix=prefix)

for blob in blobs:
    if blob.name.endswith("/"):
        continue
    filename = blob.name.replace(prefix, "")
    local_path = os.path.join(INPUTE_FOLDER, filename)
    
    blob.download_to_filename(local_path)

In [11]:
jsonl_files = glob.glob(os.path.join(
    OUTPUT_FOLDER, f"chunks-*.jsonl"))
print("Number of files to process:", len(jsonl_files))
jsonl_files

Number of files to process: 6


['RAG_outputs\\chunks-Medicine.jsonl',
 'RAG_outputs\\chunks-OB_GYN_NICU.jsonl',
 'RAG_outputs\\chunks-Outpatient_ER.jsonl',
 'RAG_outputs\\chunks-policy.jsonl',
 'RAG_outputs\\chunks-Radiology_Imaging.jsonl',
 'RAG_outputs\\chunks-Surgery.jsonl']

In [16]:
jsonl_files = glob.glob(os.path.join(
    OUTPUT_FOLDER, f"embeddings-*.jsonl"))
print("Number of files to process:", len(jsonl_files))
jsonl_files

Number of files to process: 6


['RAG_outputs\\embeddings-Medicine.jsonl',
 'RAG_outputs\\embeddings-OB_GYN_NICU.jsonl',
 'RAG_outputs\\embeddings-Outpatient_ER.jsonl',
 'RAG_outputs\\embeddings-policy.jsonl',
 'RAG_outputs\\embeddings-Radiology_Imaging.jsonl',
 'RAG_outputs\\embeddings-Surgery.jsonl']

In [12]:
for jsonl_file in jsonl_files:
    print("Processing file:", jsonl_file)
    data_df = pd.read_json(jsonl_file, lines=True)
    # drop empty chunck
    data_df = data_df[data_df["chunk"].apply(lambda x: isinstance(x, str) and x.strip() != "")]
    print("Shape:", data_df.shape)
    print(data_df.head())
    chunks = data_df["chunk"].values
    chunks = chunks.tolist()
    embeddings = generate_text_embeddings(
        chunks, EMBEDDING_DIMENSION, batch_size=15)
    data_df["embedding"] = embeddings

    time.sleep(5)

    jsonl_filename = jsonl_file.replace("chunks-", "embeddings-")
    with open(jsonl_filename, "w") as json_file:
        json_file.write(data_df.to_json(orient='records', lines=True))

Processing file: RAG_outputs\chunks-Medicine.jsonl
Shape: (118, 3)
                                               chunk      book  \
0  1\nHPI SEC & \nSSER\nPatient safety measuremen...  Medicine   
1  While healthcare holds healing without \nharm ...  Medicine   
2  The refinement logic was based on \ncapturing ...  Medicine   
3  Inherent or unavoidable suffering \nincludes t...  Medicine   
4  While deviations from performance \nstandards ...  Medicine   

                                           embedding  
0  [0.1138890833, 0.0289934035, 0.0022489615, -0....  
1  [0.10464508830000001, 0.0303351562, -0.0084878...  
2  [0.109498702, 0.0062420233, -0.0023352383, 0.0...  
3  [0.10130988810000001, 0.0219200682, 0.00183788...  
4  [0.10436295720000001, 0.0204037465, 0.01318769...  
Processing file: RAG_outputs\chunks-OB_GYN_NICU.jsonl
Shape: (1, 3)
         chunk         book                                          embedding
0  OB_GYN_NICU  OB_GYN_NICU  [0.0584158599, -0.040183480800

In [ ]:

storage_client = storage.Client()
bucket = storage_client.bucket(bucket_name)
parser = argparse.ArgumentParser(description="Command description.")

bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(source_blob_name)
pdf_bytes = blob.download_as_bytes()
pdf_stream = BytesIO(pdf_bytes)

doc = fitz.open(stream=pdf_stream, filetype="pdf")

pages = []
for page in doc:
    text = page.get_text("text")   # preserves line breaks
    # remove headers and footers
    clean_text = "\n".join([
        line for line in text.splitlines()
        if not line.strip().startswith("WHITE PAPER") and
           not line.strip().startswith("©Press Ganey")
    ])
    pages.append(clean_text.strip())

full_text = "\n\n".join(pages)
print("✅ Extracted characters:", len(full_text))


✅ Extracted characters: 178499


In [7]:
def generate_text_embeddings(chunks, dimensionality: int = 256, batch_size=250, max_retries=5, retry_delay=5):
    # Max batch size is 250 for Vertex AI
    all_embeddings = []

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]

        # Retry logic with exponential backoff
        retry_count = 0
        while retry_count <= max_retries:
            try:
                response = llm_client.models.embed_content(
                    model=EMBEDDING_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(
                        output_dimensionality=dimensionality),
                )
                all_embeddings.extend(
                    [embedding.values for embedding in response.embeddings])
                break

            except errors.APIError as e:
                retry_count += 1
                if retry_count > max_retries:
                    print(
                        f"Failed to generate embeddings after {max_retries} attempts. Last error: {str(e)}")
                    raise

                # Calculate delay with exponential backoff
                wait_time = retry_delay * (2 ** (retry_count - 1))
                print(
                    f"API error (code: {e.code}): {e.message}. Retrying in {wait_time} seconds (attempt {retry_count}/{max_retries})...")
                time.sleep(wait_time)

    return all_embeddings

In [14]:
with open("extracted_text.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print("Performing semantic chunking...")
text_splitter = SemanticChunker(
    embedding_function=generate_text_embeddings,
    breakpoint_threshold_type="percentile",  
    breakpoint_threshold_amount=60           # lower = more chunks
)

text_chunks = text_splitter.create_documents([full_text])
text_chunks = [doc.page_content for doc in text_chunks]
print("✅ Number of semantic chunks:", len(text_chunks))

Performing semantic chunking...
✅ Number of semantic chunks: 469


In [15]:
OUTPUT_FOLDER = "RAG_outputs"
chunk_file = "semantic chunks-hpi-sec-sser.pdf.jsonl"
if text_chunks is not None:
    # Save the chunks
    data_df = pd.DataFrame(text_chunks, columns=["chunk"])
    data_df["book"] = "hpi-sec-sser.pdf"
    print("Shape:", data_df.shape)
    print(data_df.head())

    jsonl_filename = os.path.join(
        OUTPUT_FOLDER, chunk_file)
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    with open(jsonl_filename, "w") as json_file:
        json_file.write(data_df.to_json(orient='records', lines=True))

Shape: (469, 2)
                                               chunk              book
0  1\nHPI SEC & \nSSER\nPatient safety measuremen...  hpi-sec-sser.pdf
1  While healthcare holds healing without \nharm ...  hpi-sec-sser.pdf
2  It serves as the foundation for \nthe calculat...  hpi-sec-sser.pdf
3  •\t\nFocus on other frequently \nencountered i...  hpi-sec-sser.pdf
4  The application of these concepts has \nbeen e...  hpi-sec-sser.pdf


In [16]:

# Get the list of chunk files
jsonl_files = glob.glob(os.path.join(
    OUTPUT_FOLDER, chunk_file))
print("Number of files to process:", len(jsonl_files))

# Process
for jsonl_file in jsonl_files:
    print("Processing file:", jsonl_file)

    data_df = pd.read_json(jsonl_file, lines=True)
    # drop empty chunck
    data_df = data_df[data_df["chunk"].apply(lambda x: isinstance(x, str) and x.strip() != "")]

    print("Shape:", data_df.shape)
    print(data_df.head())

    chunks = data_df["chunk"].values
    chunks = chunks.tolist()

    embeddings = generate_text_embeddings(
        chunks, EMBEDDING_DIMENSION, batch_size=15)
    data_df["embedding"] = embeddings

    time.sleep(5)

    # Save
    print("Shape:", data_df.shape)
    print(data_df.head())

    jsonl_filename = jsonl_file.replace("semantic chunks-", "embeddings-")
    with open(jsonl_filename, "w") as json_file:
        json_file.write(data_df.to_json(orient='records', lines=True))

Number of files to process: 1
Processing file: RAG_outputs\semantic chunks-hpi-sec-sser.pdf.jsonl
Shape: (469, 2)
                                               chunk              book
0  1\nHPI SEC & \nSSER\nPatient safety measuremen...  hpi-sec-sser.pdf
1  While healthcare holds healing without \nharm ...  hpi-sec-sser.pdf
2  It serves as the foundation for \nthe calculat...  hpi-sec-sser.pdf
3  •\t\nFocus on other frequently \nencountered i...  hpi-sec-sser.pdf
4  The application of these concepts has \nbeen e...  hpi-sec-sser.pdf
Shape: (469, 3)
                                               chunk              book  \
0  1\nHPI SEC & \nSSER\nPatient safety measuremen...  hpi-sec-sser.pdf   
1  While healthcare holds healing without \nharm ...  hpi-sec-sser.pdf   
2  It serves as the foundation for \nthe calculat...  hpi-sec-sser.pdf   
3  •\t\nFocus on other frequently \nencountered i...  hpi-sec-sser.pdf   
4  The application of these concepts has \nbeen e...  hpi-sec-sser.pdf   


In [ ]:
book_mappings = {
    "policy": {"author": "LLM", "year": 2025},
}
def load_text_embeddings(df, collection, batch_size=500):

    # Generate ids
    df["id"] = df.index.astype(str)
    hashed_books = df["book"].apply(
        lambda x: hashlib.sha256(x.encode()).hexdigest()[:16])
    df["id"] = hashed_books + "-" + df["id"]

    metadata = {
        "book": df["book"].tolist()[0]
    }
    if metadata["book"] in book_mappings:
        book_mapping = book_mappings[metadata["book"]]
        metadata["author"] = book_mapping["author"]
        metadata["year"] = book_mapping["year"]

    # Process data in batches
    total_inserted = 0
    for i in range(0, df.shape[0], batch_size):
        # Create a copy of the batch and reset the index
        batch = df.iloc[i:i+batch_size].copy().reset_index(drop=True)

        ids = batch["id"].tolist()
        documents = batch["chunk"].tolist()
        metadatas = [metadata for item in batch["book"].tolist()]
        embeddings = batch["embedding"].tolist()

        collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=embeddings
        )
        total_inserted += len(batch)
        print(f"Inserted {total_inserted} items...")

    print(
        f"Finished inserting {total_inserted} items into collection '{collection.name}'")


In [ ]:
def load(method="char-split"):
    print("load()")

    # Clear Cache
    chromadb.api.client.SharedSystemClient.clear_system_cache()

    # Connect to chroma DB
    client = chromadb.HttpClient(host=CHROMADB_HOST, port=CHROMADB_PORT)

    # Get a collection object from an existing collection, by name. If it doesn't exist, create it.
    collection_name = f"{method}-collection"
    print("Creating collection:", collection_name)

    try:
        # Clear out any existing items in the collection
        client.delete_collection(name=collection_name)
        print(f"Deleted existing collection '{collection_name}'")
    except Exception:
        print(f"Collection '{collection_name}' did not exist. Creating new.")

    collection = client.create_collection(
        name=collection_name, metadata={"hnsw:space": "cosine"})
    print(f"Created new empty collection '{collection_name}'")
    print("Collection:", collection)

    # Get the list of embedding files
    jsonl_files = glob.glob(os.path.join(
        OUTPUT_FOLDER, f"embeddings-{method}-*.jsonl"))
    print("Number of files to process:", len(jsonl_files))

    # Process
    for jsonl_file in jsonl_files:
        print("Processing file:", jsonl_file)

        data_df = pd.read_json(jsonl_file, lines=True)
        print("Shape:", data_df.shape)
        print(data_df.head())

        # Load data
        load_text_embeddings(data_df, collection)
